# Find and dock binders with MaSIF-seed

Search a processed binder database against your target and dock candidates with Open3D.

This notebook previews the configuration by default. Edit your paths and set `RUN_TASK = True` when you are ready to run.


## 1. Before you start

- Install the `design` optional dependencies.
- Obtain `weights/seed/ipa_12A_surface_atom.pt` using the [checkpoint instructions](../USAGE.md#convert-tensorflow-weights).
- Have a target packet, one or more source packets, and their referenced PDB files.

Use the `Python (masif_torch)` kernel. See [installation](../README.md#installation) and the [user guide](../USAGE.md).


## 2. Open the project


In [ ]:
from pathlib import Path
import importlib.util
import os
import shlex
import shutil
import subprocess
import sys
import tempfile

from omegaconf import OmegaConf

ROOT = next((path for path in (Path.cwd(), *Path.cwd().parents)
             if (path / "pyproject.toml").is_file() and (path / "examples").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("Open this notebook from the MaSIF_torch repository or its tutorials directory.")
os.chdir(ROOT)
CLI = [sys.executable, "-m", "masif"]
RUN_TASK = False  # Change to True only after editing the paths below.
print("Repository:", ROOT)
print("Python:", sys.executable)


## Optional: pack processed target and candidate surfaces

Skip this section if you already have the two NPZ files. Use data you prepared with
[protein preparation](prepare_protein.ipynb) or [protein–ligand preparation](prepare_ligand.ipynb).
The target needs **flipped** search descriptors; the candidate needs **straight** descriptors.
Keep the PDB files referenced by the packets available and in the same coordinate system.

The optional download supplies a small processed example. It does not supply the model weights.
Both flags below are off by default. The `--trusted-indices` option is for processed data from a source you trust.


In [ ]:
DOWNLOAD_EXAMPLE = False
PACK_SURFACES = False
example_root = Path("data/neosurf_example")
TARGET_ROOT = example_root / "with_ligand"
CANDIDATE_ROOT = example_root / "decoys"
TARGET_ID = "1A7X_A"
CANDIDATE_ID = "2GKW_B"
TARGET_PACKET = Path("data/design/target_with_ligand.npz")
CANDIDATE_PACKET = Path("data/design/source.npz")

if DOWNLOAD_EXAMPLE:
    subprocess.run([sys.executable, "tools/fetch_design_fixtures.py", "--output", str(example_root), "--download"], check=True)

if PACK_SURFACES:
    for folder in (TARGET_ROOT, CANDIDATE_ROOT):
        if not folder.is_dir():
            raise FileNotFoundError(f"Processed data not found: {folder}. Prepare your data or enable DOWNLOAD_EXAMPLE.")
    TARGET_PACKET.parent.mkdir(parents=True, exist_ok=True)
    CANDIDATE_PACKET.parent.mkdir(parents=True, exist_ok=True)
    for folder, identifier, role, packet in (
        (TARGET_ROOT, TARGET_ID, "target", TARGET_PACKET),
        (CANDIDATE_ROOT, CANDIDATE_ID, "source", CANDIDATE_PACKET),
    ):
        subprocess.run(CLI + ["pack-design", "--root", str(folder), "--identifier", identifier,
                              "--role", role, "--trusted-indices", "--output", str(packet)], check=True)
else:
    print("Packing is off. Supply existing packets or edit the processed roots and enable PACK_SURFACES.")


## 3. Choose your inputs and settings

Paths below are relative to the repository unless you provide an absolute path. Choose a separate output directory for each run.


In [ ]:
TASK = "seed"
config = OmegaConf.create({
    "task": "seed", "action": "search",
    "target": str(TARGET_PACKET), "database": [str(CANDIDATE_PACKET)],
    "checkpoint": "weights/seed/ipa_12A_surface_atom.pt",
    "output": "results/tutorial_seed", "device": "cpu", "seed": 42, "num_sites": 1,
    "registration": {"mode": "open3d", "iterations": 2000, "radius": 1.5,
                     "confidence": 0.999, "seed": 42},
})
# Add more source NPZ paths to config.database for a larger candidate database.
# Use config.action = "match" to retrieve candidates without docking them.


## 4. Preview your configuration

This prints the Hydra configuration without loading your input data or running the task.


In [ ]:
config_directory = tempfile.TemporaryDirectory(prefix="masif_tutorial_")
config_path = Path(config_directory.name) / "task.yaml"
OmegaConf.save(config, config_path)
command = CLI + [TASK, "--config", str(config_path)]
print(shlex.join(command))
subprocess.run(command + ["--cfg", "job"], check=True)


## 5. Run when ready

After editing the settings, set `RUN_TASK = True` in the setup cell and rerun the notebook. The checks below help identify missing inputs before starting.


In [ ]:
if RUN_TASK:
    from masif_torch.shared.design.data import SurfaceRecord

    for filename in [config.target, config.checkpoint, *config.database]:
        if not Path(filename).is_file():
            raise FileNotFoundError(f"Missing {filename}. Prepare the packet or obtain the checkpoint first.")
    if config.action == "search" and importlib.util.find_spec("open3d") is None:
        raise ImportError("Install the design extra from the README before docking: python -m pip install '.[design]'")
    for packet in [config.target, *config.database]:
        surface = SurfaceRecord.from_npz(packet)
        if surface.pdb_path is None or not surface.pdb_path.is_file():
            raise FileNotFoundError(f"Packet {packet} needs an accessible referenced PDB. Repack it with the PDB available.")
    subprocess.run(command, check=True)
else:
    print("Preview only. Edit your paths and set RUN_TASK = True to run.")


## 6. Next steps

Find your run in the selected `output` directory. For ligand-containing targets, continue with [MaSIF-neosurface](neosurface.ipynb). For candidate refinement, see [Rosetta](rosetta.ipynb).
